# AdwenTTS on a free GPU

Runs the AdwenTTS voice-cloning worker on Google Colab's free T4.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
2. The last cell prints a public link to a web page where you upload a reference clip, type text and download the audio.

Free Colab sessions time out after a few idle hours and when the browser closes. Nothing is lost except the running server; re-run the cells to come back.

In [ ]:
!nvidia-smi -L || echo "No GPU: switch the runtime type to T4 GPU before continuing" 

## Install

Installs the worker with the Chatterbox engine straight from GitHub, pinning PyTorch, torchvision and torchaudio to matching versions (Chatterbox requires torch 2.6.0 and Colab ships a newer torchvision that would otherwise break the import). Takes 3 to 5 minutes the first time. The last line must print `Installed adwentts_worker 0.1.0` and `cuda: True`. If Colab shows a restart banner instead, restart the runtime, skip this cell and continue below.

In [ ]:
# Chatterbox pins torch 2.6.0; torchvision/torchaudio must match or transformers fails to import.
%pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    "adwentts-worker[chatterbox,ui] @ git+https://github.com/sdanquah101/adwenTTS.git@main#subdirectory=worker"

import importlib
importlib.invalidate_caches()
import torch, torchvision, adwentts_worker
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cuda:", torch.cuda.is_available())
print("Installed adwentts_worker", adwentts_worker.__version__)

## Download the model

First run fetches about 2 GB of weights from the Hugging Face Hub and caches them for this session.

In [ ]:
import os
os.environ["ADWENTTS_ENGINE"] = "chatterbox"   # or "chatterbox-turbo" for faster, slightly lower quality
os.environ["ADWENTTS_VOICES_DIR"] = "/content/voices"

from adwentts_worker.engines import create_engine
from adwentts_worker.service import SynthesisService
from adwentts_worker.voices import VoiceStore

engine = create_engine(os.environ["ADWENTTS_ENGINE"])
engine.load()
service = SynthesisService(engine, VoiceStore(os.environ["ADWENTTS_VOICES_DIR"]))
print("Ready:", engine.name, "on", engine.device)

## Option A: web page with a public link

Upload or record a 10 to 20 second clip of the voice, type the text, click Generate, download the file. The cell keeps running while the page is live; stop it to close the link.

In [ ]:
from adwentts_worker.ui import build_ui
build_ui(service).launch(share=True)

## Option B: generate files directly in the notebook

Upload a reference clip, synthesize, and download the result.

In [ ]:
from google.colab import files
from pathlib import Path
from adwentts_worker.engines import SynthesisOptions

uploaded = files.upload()                      # pick your reference clip
ref = Path(next(iter(uploaded)))

text = """Akwaaba. Welcome to Accra, the capital of Ghana.
We are glad you could join us today."""

result = service.synthesize(text, reference_path=ref, fmt="mp3", options=SynthesisOptions(exaggeration=0.5, cfg_weight=0.5))
out = Path("speech.mp3"); out.write_bytes(result.audio)
print(f"{result.duration_seconds:.1f}s of audio in {result.elapsed_seconds:.1f}s")
files.download(str(out))

## Option C: HTTP API

Starts the REST API in the background and prints a URL that works from this browser session. Use it with the Fastify API or any HTTP client.

In [ ]:
import subprocess, time
proc = subprocess.Popen(["adwentts", "serve", "--port", "8000"])
time.sleep(5)
from google.colab.output import eval_js
print("API base URL:", eval_js("google.colab.kernel.proxyPort(8000)"))
print("Docs:", eval_js("google.colab.kernel.proxyPort(8000)") + "docs")